In [9]:
import pandas as pd
from pathlib import Path

# Busca data/ (desde eda/, tables/ o raíz PI/)
_DATA_CANDIDATES = lambda base: (base / "data", base / "Proyecto_Integrador" / "data")
DATA_DIR = next(
    (d for base in [Path.cwd(), *Path.cwd().parents]
     for d in _DATA_CANDIDATES(base)
     if (d / "bureau.parquet").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("No se encontró data/bureau.parquet en el proyecto")

df_bureau = pd.read_parquet(DATA_DIR / "bureau.parquet")

info_filas_col=df_bureau.shape

print("Cantidad de filas: ", info_filas_col[0])
print("Cantidad de columnas: ", info_filas_col[1])

Cantidad de filas:  1716428
Cantidad de columnas:  17


In [10]:
df_bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [11]:
totalFilas=len(df_bureau)
nulosBureau=(
    df_bureau.isnull().sum()
    .reset_index()
)

nulosBureau.columns=["columna","nulos"]
nulosBureau["porcentaje"]=nulosBureau["nulos"]/totalFilas
nulosBureau=nulosBureau.sort_values(by="nulos", ascending=False)
nulosBureau


,columna,nulos,porcentaje
16,AMT_ANNUITY,1226791,0.714735
8,AMT_CREDIT_MAX_OVERDUE,1124488,0.655133
7,DAYS_ENDDATE_FACT,633653,0.369170
12,AMT_CREDIT_SUM_LIMIT,591780,0.344774
11,AMT_CREDIT_SUM_DEBT,257669,0.150119
6,DAYS_CREDIT_ENDDATE,105553,0.061496
10,AMT_CREDIT_SUM,13,0.000008
0,SK_ID_CURR,0,0.000000
1,SK_ID_BUREAU,0,0.000000
5,CREDIT_DAY_OVERDUE,0,0.000000


Eliminacion de las columnas por demasiada cantidad de nulos:
    - AMT_ANNUITY con un 71% de nulos
    - AMT_CREDIT_MAX_OVERDUE y 65 % de nulos

Descripición de Variables a Analizar:
 - AMT_CREDIT_SUM_LIMIT: esta variable indica cuánto cupo o límite tiene disponible/asignado un crédito de tipo tarjeta de crédito dentro del historial reportado por el buró.

In [12]:
df_bureau_clean=df_bureau.drop(columns=["AMT_ANNUITY","AMT_CREDIT_MAX_OVERDUE"])
print("Cantidad de columnas antes de la eliminacion: ", df_bureau.shape[1])
print("Cantidad de columnas despues de la eliminacion: ", df_bureau_clean.shape[1])
print(f"Se eliminaron {len(df_bureau.columns)-df_bureau_clean.shape[1]} columnas")

Cantidad de columnas antes de la eliminacion:  17
Cantidad de columnas despues de la eliminacion:  15
Se eliminaron 2 columnas


Eliminamos las 13 fila de AMT_CREDIT_SUM que estaban nulas, porque eran insignificantes para el dataset

In [13]:
filasEliminadaas=0
filasEliminadas=filasEliminadaas+len(df_bureau_clean[df_bureau_clean["AMT_CREDIT_SUM"].isnull()])
filasAcabadasdeEliminar=len(df_bureau_clean[df_bureau_clean["AMT_CREDIT_SUM"].isnull()])
df_bureau_clean=df_bureau_clean.dropna(subset=["AMT_CREDIT_SUM"])
print(f"Se eliminaron {filasAcabadasdeEliminar} fila nulas de AMT_CREDIT_SUM")


Se eliminaron 13 fila nulas de AMT_CREDIT_SUM


Analizar si los nulos aparecen más en créditos cerrados o activos

In [14]:
limit_nulo = df_bureau_clean["AMT_CREDIT_SUM_LIMIT"].isna()

tabla_cruzada = pd.crosstab(
    df_bureau_clean["CREDIT_ACTIVE"],
    limit_nulo,
    margins=True,
    normalize="index",
) * 100

print("\nTABLA CRUZADA: CREDIT_ACTIVE VS AMT_CREDIT_SUM_LIMIT_NULO")
display(tabla_cruzada.round(2))


TABLA CRUZADA: CREDIT_ACTIVE VS AMT_CREDIT_SUM_LIMIT_NULO


AMT_CREDIT_SUM_LIMIT,False,True
CREDIT_ACTIVE,,
Active,72.52,27.48
Bad debt,57.14,42.86
Closed,61.69,38.31
Sold,24.48,75.52
All,65.52,34.48


- Tiene un 27% de nulos en Active
- Tiene un 42% de nulos en Bad debt
- Tiene un 38% de nulos en Closed
- Tiene un 75% de nulos en Sold


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = df_bureau.copy()
col = "AMT_CREDIT_SUM_LIMIT"
df["LIMIT_NULO"] = df[col].isna()
df["ES_TARJETA_CREDITO"] = df["CREDIT_TYPE"] == "Credit card"
total = len(df)
nulos = df[col].isna().sum()
no_nulos = df[col].notna().sum()
porcentaje_nulos = nulos / total * 100
print("RESUMEN GENERAL")
print("-" * 50)
print(f"Total de registros: {total:,}")
print(f"Nulos en {col}: {nulos:,}")
print(f"No nulos en {col}: {no_nulos:,}")
print(f"Porcentaje de nulos: {porcentaje_nulos:.2f}%")

RESUMEN GENERAL
--------------------------------------------------
Total de registros: 1,716,428
Nulos en AMT_CREDIT_SUM_LIMIT: 591,780
No nulos en AMT_CREDIT_SUM_LIMIT: 1,124,648
Porcentaje de nulos: 34.48%


In [16]:
total_registros = df_bureau.groupby("CREDIT_TYPE", dropna=False).size()
no_nulos=df_bureau_clean.groupby("CREDIT_TYPE",dropna=False)["AMT_CREDIT_SUM_LIMIT"].count()
nulos=total_registros-no_nulos

tabla_nulos = pd.DataFrame({
    "total_registros": total_registros,
    "nulos": nulos,
    "no_nulos": no_nulos
}).reset_index()

tabla_nulos["porcentaje_nulos"] = (
    tabla_nulos["nulos"] / tabla_nulos["total_registros"] * 100
).round(2)

tabla_nulos = tabla_nulos.sort_values("porcentaje_nulos", ascending=False)

tabla_nulos



,CREDIT_TYPE,total_registros,nulos,no_nulos,porcentaje_nulos
5,Interbank credit,1,1,0,100.00
7,Loan for purchase of shares (margin lending),4,4,0,100.00
9,Loan for working capital replenishment,469,452,17,96.38
6,Loan for business development,1975,1803,172,91.29
8,Loan for the purchase of equipment,19,17,2,89.47
13,Real estate loan,27,24,3,88.89
10,Microloan,12413,10201,2212,82.18
1,Car loan,27690,16527,11163,59.69
3,Consumer credit,1251615,439919,811696,35.15
4,Credit card,402195,117613,284582,29.24


Intentaremos Explicar mejor las categorias a ver si encontramos alguna razón por las cuales son nulos:
- Consumer credit: Crédito de consumo. Puede ser un préstamo para compras personales, electrodomésticos, tecnología, gastos familiares.
- Credit card: Tarjeta de crédito. Es un producto con cupo o límite rotativo.
- Car loan: Crédito para compra de vehículo. 
- Mortgage: Crédito hipotecario, préstamo para vivienda.
- Microloan: Microcrédito, normalmente de menor monto, usado para pequeños negocios o necesidades puntuales
- Loan for business development: Préstamo para desarrollo o crecimiento de negocio.
- Loan for working capital replenishment: Préstamo para capital de trabajo, es decir, dinero para operar un negocio: inventario, pagos, insumos, caja,
- Loan for the purchase of equipment:Préstamo para comprar maquinaria, equipos o herramientas
- Loan for purchase of shares (margin lending): Crédito para compra de acciones usando financiamiento o margen.
- Interbank credit: Crédito entre bancos o instituciones financieras
- Real estate loan: Crédito relacionado con bienes raíces, puede ser para compra, inversión o financiación inmobiliaria.
- Cash loan (non-earmarked): Préstamo en efectivo sin destinación específica. “Non-earmarked” significa que no está marcado para un uso concreto
- Mobile operator loan: Crédito asociado a operador móvil, posiblemente financiación de equipo celular o servicios.
- Another type of loan: Otro tipo de préstamo que no cae en las categorías anteriores
- Unknown type of loan: Tipo de crédito desconocido o no informado.






In [17]:
col = "AMT_CREDIT_SUM_LIMIT"
df_bureau_clean["ES_TARJETA_CREDITO"]=df_bureau_clean["CREDIT_TYPE"]=="Credit card"
total_registros = (
    df_bureau_clean
    .groupby("ES_TARJETA_CREDITO", dropna=False)
    .size()
)
no_nulos = (
    df_bureau_clean
    .groupby("ES_TARJETA_CREDITO", dropna=False)[col]
    .count()
)
nulos = total_registros - no_nulos

# 5. Crear tabla resumen
tabla_tarjeta_vs_no = pd.DataFrame({
    "total_registros": total_registros,
    "nulos": nulos,
    "no_nulos": no_nulos
}).reset_index()

# 6. Calcular porcentaje de nulos
tabla_tarjeta_vs_no["porcentaje_nulos"] = (
    tabla_tarjeta_vs_no["nulos"] / tabla_tarjeta_vs_no["total_registros"] * 100
).round(2)

# 7. Cambiar True/False por etiquetas más claras
tabla_tarjeta_vs_no["ES_TARJETA_CREDITO"] = tabla_tarjeta_vs_no["ES_TARJETA_CREDITO"].map({
    True: "Sí es tarjeta de crédito",
    False: "No es tarjeta de crédito"
})

tabla_tarjeta_vs_no




,ES_TARJETA_CREDITO,total_registros,nulos,no_nulos,porcentaje_nulos
0,No es tarjeta de crédito,1314220,474154,840066,36.08
1,Sí es tarjeta de crédito,402195,117613,284582,29.24


## Relleno de la columna AMT_CREDIT_SUM_LIMIT

Se analizó que AMT_CREDIT_SUM_LIMIT representa el límite o cupo de crédito reportado en buró, una variable especialmente interpretable en productos como tarjetas de crédito. Sin embargo, los valores nulos no se explican únicamente por si el crédito es o no tarjeta, ya que pueden existir casos particulares asociados al tipo de producto, al estado del crédito o a la falta de reporte de la información. Dado que esta variable puede aportar información valiosa sobre la capacidad crediticia del cliente, se decide conservarla e imputar los valores NaN con 0 al momento de construir agregados por cliente. Esta imputación se interpreta como ausencia de límite reportado o no aplicabilidad del campo, no necesariamente como que el cliente realmente no tenía cupo disponible.

In [18]:
df_bureau_clean["AMT_CREDIT_SUM_LIMIT"] = df_bureau_clean["AMT_CREDIT_SUM_LIMIT"].fillna(0)
df_bureau_clean["AMT_CREDIT_SUM_LIMIT"].isna().sum()

np.int64(0)

### DAYS_ENDDATE_FACT:
 - Descripción: indica cuántos días antes de la solicitud actual terminó realmente un crédito reportado en el buró.
 
 ### Ejemplo: 
  - El cliente abrió un crédito anterior el 1 de abril de 2022.
    El cliente terminó/pagó/cerró ese crédito el 1 de junio de 2024.
    Ese mismo día pidió un nuevo crédito en Home Credit.

In [19]:
col = "DAYS_ENDDATE_FACT"

# 1. Total de registros por estado del crédito
total_registros = (
    df_bureau_clean
    .groupby("CREDIT_ACTIVE", dropna=False)
    .size()
)

# 2. Cantidad de registros NO nulos en DAYS_ENDDATE_FACT
no_nulos = (
    df_bureau_clean
    .groupby("CREDIT_ACTIVE", dropna=False)[col]
    .count()
)

# 3. Cantidad de nulos
nulos = total_registros - no_nulos

# 4. Crear tabla resumen
tabla_days_enddate = pd.DataFrame({
    "total_registros": total_registros,
    "nulos": nulos,
    "no_nulos": no_nulos
}).reset_index()

# 5. Calcular porcentaje de nulos
tabla_days_enddate["porcentaje_nulos"] = (
    tabla_days_enddate["nulos"] / tabla_days_enddate["total_registros"] * 100
).round(2)

# 6. Calcular porcentaje de registros sobre toda la tabla
tabla_days_enddate["porcentaje_registros"] = (
    tabla_days_enddate["total_registros"] / len(df_bureau_clean) * 100
).round(2)

# 7. Ordenar
tabla_days_enddate = tabla_days_enddate.sort_values(
    "porcentaje_nulos",
    ascending=False
)

tabla_days_enddate

,CREDIT_ACTIVE,total_registros,nulos,no_nulos,porcentaje_nulos,porcentaje_registros
0,Active,630599,628630,1969,99.69,36.74
3,Sold,6527,4879,1648,74.75,0.38
1,Bad debt,21,11,10,52.38,0.00
2,Closed,1079268,125,1079143,0.01,62.88


# Valores Incogruentes de DAYS_ENDDATE_FACT
### - Podemos notar que la mayor cantidad de nulos se concentra los creditos activos con un 99%, lo cuál tiene mucho sentido
### - Para los están cerrados y no tiene fecha fin DAYS_ENDDATE_FACT los tomaremos como una incongruencia
### - Se identificaron 125 registros con CREDIT_ACTIVE = "Closed" y  nulo. Dado que esta variable representa la fecha real de finalización del crédito y aplica principalmente a créditos cerrados, estos casos pueden considerarse posibles inconsistencias o información incompleta. Como representan una proporción mínima frente al total de créditos cerrados, se decide eliminarlos para evitar ruido en el análisis.

In [20]:
col = "DAYS_ENDDATE_FACT"
filasEliminadaas=filasEliminadaas+len(df_bureau_clean[(df_bureau_clean["CREDIT_ACTIVE"]=="Closed")
&(df_bureau_clean["DAYS_ENDDATE_FACT"].isna())])
indiceEliminar=df_bureau_clean[(df_bureau_clean["CREDIT_ACTIVE"]=="Closed")
&(df_bureau_clean["DAYS_ENDDATE_FACT"].isna())]
df_bureau_clean=df_bureau_clean.drop(indiceEliminar.index)

#Verificar
df_bureau_clean[
    (df_bureau_clean["CREDIT_ACTIVE"] == "Closed") &
    (df_bureau_clean["DAYS_ENDDATE_FACT"].isna())
].shape



(0, 16)

# DAYS_ENDDATE_FACT: 
 
 ## el 99% de los valores nuelo son creditos activos lo cuál explica porque no tiene fecha fin 


In [21]:
import pandas as pd



df_bureau_clean.groupby('CREDIT_ACTIVE')['DAYS_ENDDATE_FACT'] \
      .apply(lambda x: x.isnull().mean() * 100) \
      .round(2) \
      .rename('pct_nulos')

CREDIT_ACTIVE
Active      99.69
Bad debt    52.38
Closed       0.00
Sold        74.75
Name: pct_nulos, dtype: float64

### Dejamos los valores nuelos que quedan, y haremos dos coumnas adicionales credit_duration y days_overun para una tabla agregada utilizando esta
#### - credit_duration= days_enddate_fact-dasy_credit

# AMT_CREDIT_SUM_DEBT
 ## - Muestra cuánto dinero sigue debiendo el cliente en ese crédito específico.
 ## - Tenemos este campo con 15% de nulos

# AMT_CREDIT_SUM:
 ## - Monto que prestó 

In [22]:
columnas = [
    "DAYS_CREDIT_ENDDATE",
    "DAYS_ENDDATE_FACT",
    "AMT_CREDIT_SUM_DEBT"
]

# Filas donde las 3 columnas son nulas al mismo tiempo
filas_tres_nulas = df_bureau_clean[
    df_bureau_clean[columnas].isna().all(axis=1)
]

# Cantidad de filas
cantidad_tres_nulas = len(filas_tres_nulas)

# Porcentaje sobre toda la tabla
porcentaje_tres_nulas = cantidad_tres_nulas / len(df_bureau_clean) * 100

print("Cantidad de filas con las 3 columnas nulas:", cantidad_tres_nulas)
print("Porcentaje sobre la tabla:", round(porcentaje_tres_nulas, 2), "%")

Cantidad de filas con las 3 columnas nulas: 13746
Porcentaje sobre la tabla: 0.8 %


In [23]:
columnas = [
    "DAYS_CREDIT_ENDDATE",
    "DAYS_ENDDATE_FACT",
    "AMT_CREDIT_SUM_DEBT"
]

# Ver cuántas filas cumplen la condición antes de eliminar
filas_a_eliminar = df_bureau_clean[df_bureau_clean[columnas].isna().all(axis=1)]

print("Filas a eliminar:", len(filas_a_eliminar))
print("Filas antes:", len(df_bureau_clean))

# Eliminar filas donde las 3 columnas son NaN al mismo tiempo
df_bureau_clean = df_bureau_clean.drop(filas_a_eliminar.index).copy()

print("Filas después:", len(df_bureau_clean))

# Verificar que ya no existan filas con las 3 columnas nulas
verificacion = df_bureau_clean[df_bureau_clean[columnas].isna().all(axis=1)]

print("Filas con las 3 columnas nulas después:", len(verificacion))

Filas a eliminar: 13746
Filas antes: 1716290
Filas después: 1702544
Filas con las 3 columnas nulas después: 0


Aquí podemos observar que los nulos siguen una lógica de negocio ya que se concentran en los creditos cerrados 

In [24]:
import pandas as pd
from pathlib import Path

_DATA_CANDIDATES = lambda base: (base / "data", base / "Proyecto_Integrador" / "data")
DATA_DIR = next(
    (d for base in [Path.cwd(), *Path.cwd().parents]
     for d in _DATA_CANDIDATES(base)
     if (d / "bureau.parquet").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("No se encontró data/bureau.parquet en el proyecto")

bur = pd.read_parquet(DATA_DIR / "bureau.parquet")
null_mask = bur['AMT_CREDIT_SUM_DEBT'].isna()
print(bur[null_mask]['CREDIT_ACTIVE'].value_counts(normalize=True))
closed = bur[bur['CREDIT_ACTIVE'] == 'Closed']
print(f"Deuda = 0 en cerrados : {(closed['AMT_CREDIT_SUM_DEBT'] == 0).mean()*100:.1f}%")
print(f"Deuda nula en cerrados: {closed['AMT_CREDIT_SUM_DEBT'].isna().mean()*100:.1f}%")

CREDIT_ACTIVE
Closed      0.700678
Active      0.285432
Sold        0.013867
Bad debt    0.000023
Name: proportion, dtype: float64
Deuda = 0 en cerrados : 82.5%
Deuda nula en cerrados: 16.7%


In [25]:
import pandas as pd



bur.groupby('CREDIT_ACTIVE')['DAYS_ENDDATE_FACT'] \
      .apply(lambda x: x.isnull().mean() * 100) \
      .round(2) \
      .rename('pct_nulos')

CREDIT_ACTIVE
Active      99.69
Bad debt    52.38
Closed       0.01
Sold        74.75
Name: pct_nulos, dtype: float64

Imputamos con 0 las filas que quedaron como nulas siguiendo la lógica de negocio

In [26]:
df_bureau_clean["AMT_CREDIT_SUM_DEBT"] = (
    df_bureau_clean["AMT_CREDIT_SUM_DEBT"].fillna(0)
)

In [27]:
# Calcular mediana
mediana_days_credit_enddate = df_bureau_clean["DAYS_CREDIT_ENDDATE"].median()

# Imputar directamente con mediana
df_bureau_clean["DAYS_CREDIT_ENDDATE"] = (
    df_bureau_clean["DAYS_CREDIT_ENDDATE"].fillna(mediana_days_credit_enddate)
)

# Verificar
df_bureau_clean["DAYS_CREDIT_ENDDATE"].isna().sum()

np.int64(0)

# Variable Agregadas


In [28]:
import numpy as np
import pandas as pd

# ── PARTE 1: agregados generales ────────────────────────────────
grupo = df_bureau_clean.groupby("SK_ID_CURR")

bureau_agg = pd.DataFrame({
    "n_credits"        : grupo["SK_ID_BUREAU"].count(),
    "n_active"         : grupo["CREDIT_ACTIVE"].apply(lambda x: (x == "Active").sum()),
    "avg_days_credit"  : grupo["DAYS_CREDIT"].mean(),
    "max_overdue_days" : grupo["CREDIT_DAY_OVERDUE"].max(),
    "total_debt"       : grupo["AMT_CREDIT_SUM_DEBT"].sum(),
    "total_credit"     : grupo["AMT_CREDIT_SUM"].sum(),
    "total_overdue_amt": grupo["AMT_CREDIT_SUM_OVERDUE"].sum(),
    "has_bad_debt"     : grupo["CREDIT_ACTIVE"].apply(lambda x: x.isin(["Bad debt", "Sold"]).any()).astype(int),
    "n_consumer"       : grupo["CREDIT_TYPE"].apply(lambda x: (x == "Consumer credit").sum()),
    "n_card"           : grupo["CREDIT_TYPE"].apply(lambda x: (x == "Credit card").sum()),
    "n_mortgage"       : grupo["CREDIT_TYPE"].apply(lambda x: (x == "Mortgage").sum()),
}).reset_index()

bureau_agg["active_pct"]  = bureau_agg["n_active"] / bureau_agg["n_credits"]
bureau_agg["has_overdue"] = (bureau_agg["max_overdue_days"] > 0).astype(int)
bureau_agg["debt_ratio"]  = np.where(
    bureau_agg["total_credit"] > 0,
    bureau_agg["total_debt"] / bureau_agg["total_credit"],
    0
)

# ── PARTE 2: agregados solo sobre créditos cerrados ─────────────
cerrados = df_bureau_clean[df_bureau_clean["CREDIT_ACTIVE"] == "Closed"].copy()

cerrados["CREDIT_DURATION"] = cerrados["DAYS_ENDDATE_FACT"] - cerrados["DAYS_CREDIT"]
cerrados["DAYS_OVERRUN"]    = cerrados["DAYS_ENDDATE_FACT"] - cerrados["DAYS_CREDIT_ENDDATE"]

agg_cerrados = cerrados.groupby("SK_ID_CURR").agg(
    mean_credit_duration = ("CREDIT_DURATION", "mean"),
    mean_days_overrun    = ("DAYS_OVERRUN",    "mean"),
    pct_paid_early       = ("DAYS_OVERRUN",    lambda x: (x > 0).mean()),
).reset_index()

# ── JOIN y relleno ───────────────────────────────────────────────
bureau_agg = bureau_agg.merge(agg_cerrados, on="SK_ID_CURR", how="left")

bureau_agg[["mean_credit_duration", "mean_days_overrun", "pct_paid_early"]] = \
    bureau_agg[["mean_credit_duration", "mean_days_overrun", "pct_paid_early"]].fillna(0)

bureau_agg.head()

,SK_ID_CURR,n_credits,n_active,avg_days_credit,max_overdue_days,total_debt,total_credit,total_overdue_amt,has_bad_debt,n_consumer,n_card,n_mortgage,active_pct,has_overdue,debt_ratio,mean_credit_duration,mean_days_overrun,pct_paid_early
0,100001,7,3,-735.000000,0,596686.5,1453365.000,0.0,0,7,0,0,0.428571,0,0.410555,228.750000,-197.0,0.250000
1,100002,8,2,-874.000000,0,245781.0,865055.565,0.0,0,4,4,0,0.250000,0,0.284122,277.000000,-163.5,0.166667
2,100003,4,1,-1400.750000,0,0.0,1017400.500,0.0,0,2,2,0,0.250000,0,0.000000,568.333333,34.0,0.333333
3,100004,2,0,-867.000000,0,0.0,189037.800,0.0,0,2,0,0,0.000000,0,0.000000,334.500000,-44.0,0.000000
4,100005,3,2,-190.666667,0,568408.5,657126.000,0.0,0,2,1,0,0.666667,0,0.864992,250.000000,5.0,1.000000


## Variables agregadas por cliente (`bureau_agg`)

Una fila por `SK_ID_CURR`. Resumen del historial crediticio en buró antes de unir con `application_train`.

| Columna | Descripción |
|---------|-------------|
| **SK_ID_CURR** | Identificador del cliente en la solicitud actual. Clave del `groupby`. |
| **n_credits** | Cantidad total de créditos reportados en buró para ese cliente. |
| **n_active** | Número de créditos con estado `Active` (vigentes al momento de la consulta). |
| **avg_days_credit** | Promedio de `DAYS_CREDIT` (días antes de la solicitud en que se abrió cada crédito). Valores más negativos = historial más antiguo. |
| **max_overdue_days** | Máximo de `CREDIT_DAY_OVERDUE` entre todos sus créditos (peor mora registrada en días). |
| **total_debt** | Suma de `AMT_CREDIT_SUM_DEBT` (deuda total reportada en buró). |
| **total_credit** | Suma de `AMT_CREDIT_SUM` (monto total de créditos otorgados en el historial). |
| **total_overdue_amt** | Suma de `AMT_CREDIT_SUM_OVERDUE` (monto total en mora/vencido). |
| **has_bad_debt** | `1` si el cliente tuvo al menos un crédito en estado `Bad debt` o `Sold`; `0` si no. Señal de alto riesgo. |
| **n_consumer** | Cantidad de créditos tipo `Consumer credit`. |
| **n_card** | Cantidad de créditos tipo `Credit card`. |
| **n_mortgage** | Cantidad de créditos tipo `Mortgage`. |
| **active_pct** | Proporción de créditos activos: `n_active / n_credits`. Mide carga crediticia vigente. |
| **has_overdue** | `1` si `max_overdue_days > 0` (alguna mora en el historial); `0` si nunca. |
| **debt_ratio** | Relación deuda / crédito: `total_debt / total_credit`. Si `total_credit` es 0, se asigna 0. |

Comparativa de antes y después

In [29]:
filas_cols_antes=df_bureau.shape
filas_antes=filas_cols_antes[0]
cols_antes=filas_cols_antes[1]
filas_cols_despues=df_bureau_clean.shape
filas_despues=filas_cols_despues[0]
cols_despues=filas_cols_despues[1]
print(f"Habian {filas_antes} filas antes y quedaron {filas_despues} diferencia de {filas_antes-filas_despues}")
print(f"Habian {cols_antes} columnas antes y quedaron {cols_despues} diferencia de {cols_antes-cols_despues}")

Habian 1716428 filas antes y quedaron 1702544 diferencia de 13884
Habian 17 columnas antes y quedaron 16 diferencia de 1
